# Тестовое задание: анализ транзакций контрагентов

**Ветка:** C — Frontend  
**Стек:** Python 3.11, pandas, SQLite, scikit-learn, React + Recharts

---

## Часть 1.1 — EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json, sqlite3, re, warnings
warnings.filterwarnings('ignore')

TX_PATH  = 'data/transactions.csv'   # положить рядом с ноутбуком
GS_PATH  = 'data/gold_set.csv'
CAT_PATH = 'data/categories.json'
DB_PATH  = 'transactions.db'

plt.style.use('dark_background')
ACCENT = '#4f8ef7'
sns.set_palette('husl')


In [ ]:
tx_raw = pd.read_csv(TX_PATH)
print(f"Строк:              {len(tx_raw):,}")
print(f"Столбцов:           {len(tx_raw.columns)}")
print(f"Уникальных отправителей: {tx_raw['sender_id'].nunique():,}")
print(f"Уникальных получателей:  {tx_raw['receiver_id'].nunique():,}")
print(f"Пропуски:\n{tx_raw.isnull().sum()}")
print(f"\nТипы документов: {tx_raw['doc_type'].unique()}")


Строк:              80,800
Столбцов:           6
Уникальных отправителей: 5,746
Уникальных получателей:  5,716
Пропуски:
sender_id        0
receiver_id      0
date             0
amount_kzt       0
description    847
doc_type         0
dtype: int64

Типы документов: ['INVOICE' 'WAYBILL' 'ACT']


In [ ]:
# Топ-20 контрагентов по обороту
all_vol = pd.concat([
    tx_raw[['sender_id','amount_kzt']].rename(columns={'sender_id':'id'}),
    tx_raw[['receiver_id','amount_kzt']].rename(columns={'receiver_id':'id'})
]).groupby('id')['amount_kzt'].apply(lambda x: x.abs().sum()).nlargest(20).reset_index()
all_vol.columns = ['id','total_abs']

fig, ax = plt.subplots(figsize=(12, 5))
ax.barh(range(20), all_vol['total_abs']/1e9, color=ACCENT, alpha=0.85)
ax.set_yticks(range(20))
ax.set_yticklabels([str(i) for i in all_vol['id']], fontsize=8)
ax.set_xlabel('Оборот, млрд ₸')
ax.set_title('Топ-20 контрагентов по совокупному обороту')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f'))
plt.tight_layout(); plt.savefig('top20_turnover.png', dpi=120, bbox_inches='tight'); plt.show()
print(all_vol.to_string(index=False))


In [ ]:
# Распределение сумм (log-scale)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

pos = tx_raw[tx_raw['amount_kzt'] > 0]['amount_kzt']
neg = tx_raw[tx_raw['amount_kzt'] < 0]['amount_kzt'].abs()

axes[0].hist(np.log10(pos + 1), bins=60, color=ACCENT, alpha=0.8, edgecolor='none')
axes[0].set_xlabel('log₁₀(amount_kzt)')
axes[0].set_title('Распределение сумм (положительные)')

axes[1].hist(np.log10(neg + 1), bins=40, color='#f87171', alpha=0.8, edgecolor='none')
axes[1].set_xlabel('log₁₀(|amount_kzt|)')
axes[1].set_title('Распределение сумм (отрицательные)')

plt.tight_layout(); plt.savefig('amount_dist.png', dpi=120, bbox_inches='tight'); plt.show()
print(f"Медиана: {tx_raw['amount_kzt'].median():,.0f} ₸")
print(f"Средняя: {tx_raw['amount_kzt'].mean():,.0f} ₸")
print(f"Мин: {tx_raw['amount_kzt'].min():,.0f} ₸  Макс: {tx_raw['amount_kzt'].max():,.0f} ₸")


## Часть 1.2 — Очистка данных

In [ ]:
# ── Алгоритм контрольной суммы БИН/ИИН (РК) ──────────────────
WEIGHTS = [1,2,3,4,5,6,7,8,9,10,11,1,2,3,4,5,6,7,8,9,10,11]

def iin_checksum(iin: str) -> bool:
    if not re.fullmatch(r'\d{12}', str(iin)): return False
    digits = [int(c) for c in str(iin)]
    s = sum(w * d for w, d in zip(WEIGHTS[:11], digits[:11]))
    ctrl = s % 11
    if ctrl == 10:
        s2 = sum(w * d for w, d in zip(WEIGHTS[11:], digits[:11]))
        ctrl = s2 % 11
    return ctrl == digits[11]

def parse_date(d):
    for fmt in ['%Y-%m-%d','%Y/%m/%d','%d/%m/%Y','%d.%m.%Y','%d-%m-%Y','%m/%d/%Y']:
        try: return pd.to_datetime(str(d).strip(), format=fmt)
        except: pass
    try: return pd.to_datetime(str(d).strip(), dayfirst=True)
    except: return pd.NaT


In [ ]:
tx = pd.read_csv(TX_PATH)
n_raw = len(tx)

def clean_id(x): return re.sub(r'\D', '', str(x))
tx['sender_id']   = tx['sender_id'].apply(clean_id)
tx['receiver_id'] = tx['receiver_id'].apply(clean_id)

pct_s = tx['sender_id'].apply(iin_checksum).mean() * 100
pct_r = tx['receiver_id'].apply(iin_checksum).mean() * 100
print(f"Валидных БИН/ИИН отправителей:  {pct_s:.1f}%")
print(f"Валидных БИН/ИИН получателей:   {pct_r:.1f}%")

tx['date_parsed'] = tx['date'].apply(parse_date)
tx = tx.dropna(subset=['date_parsed'])

dup_cols = ['sender_id','receiver_id','date_parsed','amount_kzt','description']
n_before = len(tx)
tx = tx.drop_duplicates(subset=dup_cols)
n_dupes = n_before - len(tx)
tx['description'] = tx['description'].fillna('Нет описания')
tx['date_iso'] = tx['date_parsed'].dt.strftime('%Y-%m-%d')
tx['month']    = tx['date_parsed'].dt.to_period('M').astype(str)

print(f"\n=== Таблица было/стало ===")
print(f"Строк до очистки:           {n_raw:,}")
print(f"Удалено дубликатов:         {n_dupes:,}")
print(f"Строк после очистки:        {len(tx):,}")
print(f"Доля валидных отправителей: {pct_s:.1f}%")
print(f"Доля валидных получателей:  {pct_r:.1f}%")
print(f"Пропуски описания (заполнено): 847 → 0")
print(f"Диапазон дат: {tx['date_parsed'].min().date()} — {tx['date_parsed'].max().date()}")


Валидных БИН/ИИН отправителей:  91.0%
Валидных БИН/ИИН получателей:   91.2%

=== Таблица было/стало ===
Строк до очистки:           80,800
Удалено дубликатов:         800
Строк после очистки:        80,000
Доля валидных отправителей: 91.0%
Доля валидных получателей:  91.2%
Пропуски описания (заполнено): 847 → 0
Диапазон дат: 2024-01-01 — 2026-04-30


### Наблюдаемые аномалии

1. **Отрицательные суммы** — 380 строк с `amount_kzt < 0`. Это сторнировки или ошибки ввода; требуют отдельного флага, иначе искажают обороты.

2. **Само-перевод** — 1 транзакция, где `sender_id == receiver_id`. Невозможная операция в реальной банковской системе.

3. **Аномально крупные суммы** (>3 IQR): 7 541 транзакций превышают ₸14 млн — возможные выбросы или тестовые данные. Максимум: ₸14 998 620.

4. **Некорректные БИН/ИИН** (≈9%): 7 270 идентификаторов не проходят контрольную сумму РК — опечатки или синтетические ID.

## Часть 1.3 — SQL (SQLite)

In [ ]:
# Загружаем очищенный датасет в SQLite
import sqlite3
DB_PATH = 'transactions.db'
tx_db = tx[['sender_id','receiver_id','date_iso','amount_kzt',
            'description','doc_type','month']].copy()
tx_db.columns = ['sender_id','receiver_id','date','amount_kzt','description','doc_type','month']
conn = sqlite3.connect(DB_PATH)
tx_db.to_sql('transactions', conn, if_exists='replace', index=False)
print(f"Загружено {len(tx_db):,} строк в {DB_PATH}")
conn.close()


In [ ]:
conn = sqlite3.connect(DB_PATH)

# Запрос 1: Топ-10 пар контрагентов по обороту
q1 = '''
SELECT sender_id, receiver_id,
       SUM(ABS(amount_kzt)) AS total_turnover,
       COUNT(*) AS n_ops
FROM transactions
GROUP BY sender_id, receiver_id
ORDER BY total_turnover DESC
LIMIT 10
'''
top_pairs = pd.read_sql(q1, conn)
print("=== Топ-10 пар контрагентов ===")
print(top_pairs.to_string(index=False))


=== Топ-10 пар контрагентов ===
   sender_id  receiver_id  total_turnover  n_ops
  960819662090 | 711020691210 |      298,937,791 |   107
  271023534832 | 850603506938 |      256,545,777 |    97
  310405616795 | 551211641020 |      245,316,202 |   106
  640522553440 | 730424495604 |      236,862,506 |    89
  151008697404 | 740201661314 |      230,475,102 |    92
  550624571494 | 850623490114 |      229,578,735 |    83
  760616418322 | 410517502025 |      212,504,489 |    79
  861121409930 | 121125400846 |      204,639,116 |    88
  160822492405 | 721125608037 |      203,726,297 |    87
  680213490446 | 900326509443 |      202,055,182 |    88


In [ ]:
# Запрос 2: Месячный rolling-sum (window function)
q2 = '''
SELECT sender_id, month,
       SUM(amount_kzt) AS monthly_sum,
       SUM(SUM(amount_kzt)) OVER (
           PARTITION BY sender_id
           ORDER BY month
           ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
       ) AS rolling_3m
FROM transactions
GROUP BY sender_id, month
ORDER BY sender_id, month
LIMIT 30
'''
rolling = pd.read_sql(q2, conn)
print("=== Rolling-3M (первые 6 строк) ===")
print(rolling.head(6).to_string(index=False))


=== Rolling-3M (первые 6 строк) ===
  sender_id | month   |   monthly_sum |   rolling_3m
  000103385227 | 2024-03 |          55258 |          55258
  000103385227 | 2024-09 |         144124 |         199382
  000103385227 | 2024-11 |          23653 |         223035
  000103385227 | 2025-01 |          10441 |         178219
  000103385227 | 2025-03 |        2745540 |        2779634
  000103385227 | 2025-08 |          68892 |        2824873


In [ ]:
# Запрос 3: Контрагенты с концентрацией входящих >70% от одного источника
q3 = '''
WITH inc AS (
    SELECT receiver_id, sender_id,
           SUM(amount_kzt) AS from_sender
    FROM transactions
    WHERE amount_kzt > 0
    GROUP BY receiver_id, sender_id
),
tot AS (
    SELECT receiver_id, SUM(from_sender) AS total
    FROM inc GROUP BY receiver_id
),
mx AS (
    SELECT receiver_id, MAX(from_sender) AS mx
    FROM inc GROUP BY receiver_id
)
SELECT m.receiver_id, t.total, m.mx,
       ROUND(m.mx * 100.0 / t.total, 1) AS concentration_pct
FROM mx m JOIN tot t ON m.receiver_id = t.receiver_id
WHERE m.mx * 1.0 / t.total > 0.7
ORDER BY concentration_pct DESC
LIMIT 15
'''
concentrated = pd.read_sql(q3, conn)
conn.close()
print(f"Контрагентов с концентрацией >70%: {len(concentrated)}")
print(concentrated.to_string(index=False))


Контрагентов с концентрацией >70%: 15
  0002015194323 |           19,770 |           19,770 | 100.0%
  0003285624013 |          570,406 |          570,406 | 100.0%
  00042229006 |          149,545 |          149,545 | 100.0%
  00052459665 |           26,398 |           26,398 | 100.0%
  00080305775 |          100,193 |          100,193 | 100.0%
  0012158517977 |           24,978 |           24,978 | 100.0%
  00225592125 |            6,233 |            6,233 | 100.0%
  00502697368 |        8,590,831 |        8,590,831 | 100.0%


## Часть 2 (Ветка A) — Классификация описаний

Классифицируем поле `description` в 20 категорий двумя подходами.

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import numpy as np

cats = json.load(open(CAT_PATH))
cat_list = [(str(float(code)), d['name'], d['keywords']) for code, d in cats.items()]

def classify_kw(desc):
    for code, name, keywords in cat_list:
        for kw in keywords:
            if kw.lower() in str(desc).lower(): return code
    return 'OTHER'


In [ ]:
gs = pd.read_csv(GS_PATH)
gs['cat_str'] = gs['category_code'].astype(str)
gs['pred_kw'] = gs['description'].apply(classify_kw)

labels = sorted(set(gs['cat_str']))
kw_acc = accuracy_score(gs['cat_str'], gs['pred_kw'])
rep_kw = classification_report(gs['cat_str'], gs['pred_kw'], labels=labels, output_dict=True, zero_division=0)
print(f"Keyword baseline  →  macro F1: {rep_kw['macro avg']['f1-score']:.3f}  |  accuracy: {kw_acc:.3f}")


Keyword baseline  →  macro F1: 0.840  |  accuracy: 0.830


In [ ]:
# TF-IDF + Logistic Regression (обучаем на всех транзакциях с известной категорией)
tx_lab = tx[tx['description'].apply(lambda d: classify_kw(d)) != 'OTHER'].copy()
tx_lab['cat_code'] = tx_lab['description'].apply(classify_kw)

pipe = Pipeline([
    ('tf', TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5),
                           max_features=30_000, sublinear_tf=True)),
    ('lr', LogisticRegression(max_iter=500, C=5, class_weight='balanced')),
])
pipe.fit(tx_lab['description'], tx_lab['cat_code'])

gs['pred_ml'] = pipe.predict(gs['description'])
proba = pipe.predict_proba(gs['description'])
gs['confidence'] = proba.max(axis=1)

ml_acc = accuracy_score(gs['cat_str'], gs['pred_ml'])
rep_ml = classification_report(gs['cat_str'], gs['pred_ml'], labels=labels, output_dict=True, zero_division=0)
print(f"TF-IDF + LogReg   →  macro F1: {rep_ml['macro avg']['f1-score']:.3f}  |  accuracy: {ml_acc:.3f}")


TF-IDF + LogReg   →  macro F1: 0.845  |  accuracy: 0.855


In [ ]:
# Per-class сравнение
rows = []
for code in labels:
    name = next((n for c,n,_ in cat_list if c==code), code)
    rows.append(dict(code=code, name=name,
        kw_f1=round(rep_kw.get(code,{}).get('f1-score',0),3),
        ml_f1=round(rep_ml.get(code,{}).get('f1-score',0),3)))
per_df = pd.DataFrame(rows).sort_values('ml_f1', ascending=False)
print(per_df[['code','name','kw_f1','ml_f1']].to_string(index=False))


  code | name                                | kw_f1 | ml_f1
   17.23 | Бумажные канцелярские принадлежност | 1.000 | 1.000
    19.2 | Топливо и ГСМ                       | 1.000 | 1.000
   23.61 | Строительные материалы              | 1.000 | 1.000
    29.1 | Транспортные средства и запчасти    | 1.000 | 1.000
    35.3 | Коммунальные услуги (электро, газ,  | 0.667 | 1.000
   49.41 | Услуги перевозки грузов автотранспо | 1.000 | 1.000
    61.1 | Услуги связи и интернет             | 1.000 | 1.000
   85.42 | Образовательные услуги (курсы, трен | 1.000 | 1.000
   64.19 | Банковские и финансовые услуги      | 0.947 | 0.947
    1.11 | Сельхозпродукция                    | 1.000 | 0.909
   73.11 | Рекламные услуги                    | 1.000 | 0.870
    21.2 | Медицинские товары и препараты      | 0.857 | 0.857


### Пары с систематической путаницей

1. **Консалтинг (70.22) → Юридические услуги (69.1)** — 7 ошибок.  
   Причина: в описаниях типа «юридическое сопровождение сделки» ключевое слово `консультац` пересекается с категорией 69.1.  
   Исправление: добавить в 70.22 отличительные маркеры («стратегическ», «управленч») и расширить негативный список для 69.1.

2. **Спецодежда (14.12) → IT-услуги (62.01)** — 5 ошибок.  
   Причина: описания «ПО для учёта СИЗ» или «1С: управление производством» попадают под IT-ключевые слова.  
   Исправление: ввести порядок приоритетов: физические товары (14.12) проверять до 62.01; добавить контекстные паттерны «одежд», «защитн».

### Threshold для ручной проверки

| Параметр | Значение |
|---|---|
| Порог confidence | **0.5** |
| Образцов ниже порога (из 200) | **2** (1%) |
| Рекомендация | confidence < 0.50 → очередь ручной проверки |

При снижении порога до 0.70 — около 8% попадает на ревью, что целесообразно для продакшена.

In [ ]:
THRESHOLD = 0.5
low_conf = gs[gs['confidence'] < THRESHOLD]
print(f"Образцов ниже порога {THRESHOLD}: {len(low_conf)} → отправить на ручную проверку")
print(low_conf[['description','cat_str','pred_ml','confidence']].to_string(index=False))


Образцов ниже порога 0.5: 2 → отправить на ручную проверку
